In [1]:
#import libraries

import os
import warnings
import pandas as pd
from dotenv import load_dotenv
from groq import Groq

warnings.filterwarnings("ignore")
load_dotenv()

True

In [2]:
#initialize groq client

client=Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("groq client initialized")

groq client initialized


In [3]:
#load graph relationships dataset

graph_relationships_df=pd.read_csv(
    "../Data/graph_relationships.csv"
)

print(
    graph_relationships_df.shape
)

display(
    graph_relationships_df.head()
)

(3483, 3)


,source,target,relationship
0,dizziness,benign paroxysmal positional vertigo,RELATED_TO
1,dizziness,bppv,RELATED_TO
2,dizziness,peripheral vertigo,RELATED_TO
3,giddiness,benign paroxysmal positional vertigo,RELATED_TO
4,giddiness,bppv,RELATED_TO


In [4]:
#convert dataframe to list

graph_relationships=graph_relationships_df.to_dict(
    orient="records"
)

print(
    len(graph_relationships)
)

3483


In [5]:
#validation prompt builder

def build_validation_prompt(relation):

    prompt=f"""

    You are an expert clinical relationship validation agent.

    Your task is to evaluate how clinically relevant
    and medically meaningful the following relationship is.

    Relationship:

    Symptom: {relation['source']}

    Relation: {relation['relationship']}

    Target: {relation['target']}

    Validation Guidelines:

    1. Focus on common clinical reasoning.
    2. Ignore weak theoretical possibilities.
    3. Prioritize primary medical associations.
    4. Reduce hallucination risk.
    5. Consider practical diagnostic relevance.
    6. Rare or indirect relationships should receive lower relevance.
    7. Strongly established medical associations should receive higher relevance.

    Clinical Relevance Levels:

    VERY_HIGH:
    Strong and highly common clinical relationship.

    HIGH:
    Common and medically meaningful relationship.

    MEDIUM:
    Possible relationship but context dependent.

    LOW:
    Weak or uncommon association.

    VERY_LOW:
    Rare, indirect, or clinically insignificant association.

    Output Format:

    Clinical_Relevance:
    VERY_HIGH or HIGH or MEDIUM or LOW or VERY_LOW

    Reason:
    Short clinical explanation.

    """

    return prompt

In [6]:
#validate single relationship

def validate_relationship(relation):

    prompt=build_validation_prompt(
        relation
    )

    response=client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[

            {
                "role":"system",
                "content":"You are an expert healthcare validation agent."
            },

            {
                "role":"user",
                "content":prompt
            }

        ],

        temperature=0

    )

    validation_result=response.choices[
        0
    ].message.content

    return validation_result

In [7]:
#test single validation

test_relation={

    "source":"dizziness",

    "relationship":"RELATED_TO",

    "target":"cancer"

}

result=validate_relationship(
    test_relation
)

print(result)

Clinical_Relevance: LOW

Reason: Dizziness is a non-specific symptom that can be associated with various conditions, and while it can be a symptom of certain types of cancer, such as brain tumors or cancer that has metastasized to the brain, it is not a primary or highly specific indicator of cancer. In most cases, dizziness is more commonly related to other conditions such as inner ear problems, dehydration, or cardiovascular issues. Therefore, the relationship between dizziness and cancer is weak and not highly clinically relevant.


In [8]:
#store relationships based on clinical relevance

very_high_relationships=[]

high_relationships=[]

medium_relationships=[]

low_relationships=[]

very_low_relationships=[]

In [9]:
#run clinical validation pipeline
for relation in graph_relationships:
    validation_output=validate_relationship(
        relation
    )
    print("\n")
    print(relation)
    print(validation_output)
    relationship_data={
        "source":relation["source"],
        "relationship":relation["relationship"],
        "target":relation["target"],
        "validation":validation_output
    }
    clinical_score=validation_output.upper()
    if "VERY_HIGH" in clinical_score:
        very_high_relationships.append(
            relationship_data
        )
    elif (
        "HIGH" in clinical_score
        and
        "VERY_HIGH" not in clinical_score
    ):
        high_relationships.append(
            relationship_data
        )

    elif "MEDIUM" in clinical_score:
        medium_relationships.append(
            relationship_data
        )

    elif (
        "LOW" in clinical_score
        and
        "VERY_LOW" not in clinical_score
    ):

        low_relationships.append(
            relationship_data
        )

    elif "VERY_LOW" in clinical_score:
        very_low_relationships.append(
            relationship_data
        )



{'source': 'dizziness', 'target': 'benign paroxysmal positional vertigo', 'relationship': 'RELATED_TO'}
Clinical_Relevance: HIGH

Reason: Dizziness is a common symptom of benign paroxysmal positional vertigo (BPPV), a vestibular disorder that affects the inner ear. The relationship between dizziness and BPPV is well-established, as BPPV is a leading cause of vertigo, which is a type of dizziness characterized by a spinning sensation. In clinical practice, patients presenting with dizziness are often evaluated for BPPV, and the diagnosis is typically confirmed with a Dix-Hallpike maneuver or roll test. The association between dizziness and BPPV is strong and medically meaningful, making it a HIGH clinical relevance relationship.


{'source': 'dizziness', 'target': 'bppv', 'relationship': 'RELATED_TO'}
Clinical_Relevance: HIGH

Reason: Dizziness is a common symptom associated with Benign Paroxysmal Positional Vertigo (BPPV), a vestibular disorder that affects the inner ear. The relatio

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kehvnsade7ts1bfbq5j0ew8m` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99765, Requested 420. Please try again in 2m39.84s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [10]:
#print very high clinical relevance

print("\nVERY HIGH RELATIONSHIPS\n")

for relation in very_high_relationships:

    print(relation)

    print("\n")


VERY HIGH RELATIONSHIPS

{'source': 'blood in motion', 'relationship': 'RELATED_TO', 'target': 'viral diarrhea', 'validation': 'Clinical_Relevance: HIGH\n\nReason: Blood in motion, also known as hematochezia or rectal bleeding, can be a symptom of various gastrointestinal disorders, including viral diarrhea. Certain viral infections that cause diarrhea, such as norovirus or rotavirus, can lead to inflammation and bleeding in the gastrointestinal tract, resulting in blood in the stool. This relationship is common and medically meaningful, as rectal bleeding is a potential complication of severe viral diarrhea. However, it is not the most common symptom, and other causes of rectal bleeding are more frequent, which is why the relevance is HIGH rather than VERY_HIGH.'}


{'source': 'pain', 'relationship': 'RELATED_TO', 'target': 'osteoarthritis', 'validation': 'Clinical_Relevance: VERY_HIGH\nReason: Pain is a primary and highly common symptom of osteoarthritis, a degenerative joint diseas

In [11]:
#print high clinical relevance

print("\nHIGH RELATIONSHIPS\n")

for relation in high_relationships:

    print(relation)

    print("\n")


HIGH RELATIONSHIPS

{'source': 'dizziness', 'relationship': 'RELATED_TO', 'target': 'benign paroxysmal positional vertigo', 'validation': 'Clinical_Relevance: HIGH\n\nReason: Dizziness is a common symptom of benign paroxysmal positional vertigo (BPPV), a vestibular disorder that affects the inner ear. The relationship between dizziness and BPPV is well-established, as BPPV is a leading cause of vertigo, which is a type of dizziness characterized by a spinning sensation. In clinical practice, patients presenting with dizziness are often evaluated for BPPV, and the diagnosis is typically confirmed with a Dix-Hallpike maneuver or roll test. The association between dizziness and BPPV is strong and medically meaningful, making it a HIGH clinical relevance relationship.'}


{'source': 'dizziness', 'relationship': 'RELATED_TO', 'target': 'bppv', 'validation': 'Clinical_Relevance: HIGH\n\nReason: Dizziness is a common symptom associated with Benign Paroxysmal Positional Vertigo (BPPV), a vest

In [12]:
#print low clinical relevance

print("\nLOW RELATIONSHIPS\n")

for relation in low_relationships:

    print(relation)

    print("\n")


LOW RELATIONSHIPS

{'source': 'nausea', 'relationship': 'RELATED_TO', 'target': 'bppv', 'validation': "Clinical Relevance: LOW\n\nReason: Nausea can be associated with many conditions, and while it's possible for patients with Benign Paroxysmal Positional Vertigo (BPPV) to experience nausea, especially during vertigo episodes triggered by specific head movements, the relationship is not as direct or exclusive as it is with other symptoms like vertigo or dizziness. Nausea in BPPV is more of a secondary symptom and can be seen in a wide range of other conditions, making the nausea-BPPV relationship less specific and less commonly a primary diagnostic consideration compared to other symptoms."}


{'source': 'benign paroxysmal positional vertigo', 'relationship': 'TREATED_BY', 'target': 'panadol', 'validation': 'Clinical_Relevance: LOW\n\nReason: Benign paroxysmal positional vertigo (BPPV) is typically treated with canalith repositioning procedures (e.g., Epley maneuver) or vestibular reh

In [13]:
#clinical validation statistics

print("\nTOTAL RELATIONSHIPS")

print(
    len(graph_relationships)
)

print("\nVERY HIGH")

print(
    len(very_high_relationships)
)

print("\nHIGH")

print(
    len(high_relationships)
)

print("\nMEDIUM")

print(
    len(medium_relationships)
)

print("\nLOW")

print(
    len(low_relationships)
)

print("\nVERY LOW")

print(
    len(very_low_relationships)
)


TOTAL RELATIONSHIPS
3483

VERY HIGH
17

HIGH
86

MEDIUM
17

LOW
33

VERY LOW
24


In [14]:
#create trusted graph context

trusted_graph_context=[]

In [15]:
#add very high relationships

for relation in very_high_relationships:

    graph_text=f"""

    {relation['source']}
    {relation['relationship']}
    {relation['target']}

    Clinical Confidence:
    VERY_HIGH

    """

    trusted_graph_context.append(
        graph_text
    )

In [16]:
#add high relationships

for relation in high_relationships:

    graph_text=f"""

    {relation['source']}
    {relation['relationship']}
    {relation['target']}

    Clinical Confidence:
    HIGH

    """

    trusted_graph_context.append(
        graph_text
    )

In [17]:
#optionally add medium relationships

for relation in medium_relationships:

    graph_text=f"""

    {relation['source']}
    {relation['relationship']}
    {relation['target']}

    Clinical Confidence:
    MEDIUM

    """

    trusted_graph_context.append(
        graph_text
    )

In [18]:
#print trusted graph context

print("\nTRUSTED GRAPH CONTEXT\n")

for item in trusted_graph_context:

    print(item)

    print("\n")


TRUSTED GRAPH CONTEXT



    blood in motion
    RELATED_TO
    viral diarrhea

    Clinical Confidence:
    VERY_HIGH

    




    pain
    RELATED_TO
    osteoarthritis

    Clinical Confidence:
    VERY_HIGH

    




    infection
    TREATED_BY
    medication

    Clinical Confidence:
    VERY_HIGH

    




    yeast infection
    TREATED_BY
    medication

    Clinical Confidence:
    VERY_HIGH

    




    herpes outbreak
    TREATED_BY
    medication

    Clinical Confidence:
    VERY_HIGH

    




    uti
    TREATED_BY
    medication

    Clinical Confidence:
    VERY_HIGH

    




    pain
    RELATED_TO
    cancer

    Clinical Confidence:
    VERY_HIGH

    




    heartburn
    RELATED_TO
    gastroesophageal reflux disease

    Clinical Confidence:
    VERY_HIGH

    




    lockjaw symptoms
    RELATED_TO
    tetanus

    Clinical Confidence:
    VERY_HIGH

    




    spasms
    RELATED_TO
    tetanus

    Clinical Confidence:
    VERY_HIGH

    




    painf

In [19]:
#create dataframe for each relevance level
import pandas as pd
very_high_df=pd.DataFrame(
    very_high_relationships
)

high_df=pd.DataFrame(
    high_relationships
)

medium_df=pd.DataFrame(
    medium_relationships
)

low_df=pd.DataFrame(
    low_relationships
)

very_low_df=pd.DataFrame(
    very_low_relationships
)

In [20]:
#save high relationships

high_df.to_csv(
    "../Data/high_relationships.csv",
    index=False
)

print("high relationships saved")

high relationships saved


In [21]:
#save very high relationships

very_high_df.to_csv(
    "../Data/very_high_relationships.csv",
    index=False
)

print("very high relationships saved")

very high relationships saved


In [22]:
#save medium relationships

medium_df.to_csv(
    r"../Data/medium_relationships.csv",
    index=False
)

print("medium relationships saved")

medium relationships saved


In [23]:
#save low relationships

low_df.to_csv(
    r"../Data/low_relationships.csv",
    index=False
)

print("low relationships saved")

low relationships saved


In [24]:
#save very low relationships

very_low_df.to_csv(
    r"../Data/very_low_relationships.csv",
    index=False
)

print("very low relationships saved")

very low relationships saved


In [25]:
#final validation summary

print("\nclinical validation completed")

print("\nmedical graph quality improved")

print("\nrelationship confidence scoring completed")

print("\nunsafe medical relationships identified")

print("\ntrusted graph context created successfully")


clinical validation completed

medical graph quality improved

relationship confidence scoring completed

unsafe medical relationships identified

trusted graph context created successfully
